In [ ]:
#used for face detection and cropping from our dataset images 
%pip install retina-face opencv-python tqdm

In [8]:
# importing libraries
import os
import cv2
from tqdm import tqdm
from retinaface import RetinaFace

In [9]:
#root path
ROOT = "dataset_split"
SPLITS = [
    "train",
    "test"
]
SUBSETS = [
    "gallery",
    "probe"
]

In [10]:
# creating output folders
for split in SPLITS:
    os.makedirs(
        os.path.join(
            ROOT,
            split,
            "gallery_cropped"
        ),
        exist_ok=True
    )

    os.makedirs(
        os.path.join(
            ROOT,
            split,
            "probe_cropped"
        ),
        exist_ok=True
    )

print("Output folders ready.")

Output folders ready.


In [11]:
# cropping and resizing the detected image
def crop_and_resize_face(image_path):
    try:
        img = cv2.imread(image_path)
        if img is None:
            return None
        faces = RetinaFace.detect_faces(
            image_path
        )

        if not isinstance(faces, dict):
            return None
        largest_area = -1
        best_box = None

        for face_id in faces:
            x1, y1, x2, y2 = (
                faces[face_id]["facial_area"]
            )
            area = (
                (x2 - x1)
                *
                (y2 - y1)
            )
            if area > largest_area:
                largest_area = area
                best_box = (
                    x1,
                    y1,
                    x2,
                    y2
                )

        if best_box is None:
            return None
        x1, y1, x2, y2 = best_box
        h, w = img.shape[:2]

        x1 = max(0, x1)
        y1 = max(0, y1)

        x2 = min(w, x2)
        y2 = min(h, y2)

        face = img[
            y1:y2,
            x1:x2
        ]
        face = cv2.resize(
            face,
            (112, 112)
        )
        return face

    except Exception as e:
        print(f"ERROR: {image_path}")
        print(e)
        return None

In [ ]:
# processing and saving all images
for split in SPLITS:
    for subset in SUBSETS:

        input_root = os.path.join(
            ROOT,
            split,
            subset
        )

        output_root = os.path.join(
            ROOT,
            split,
            subset + "_cropped"
        )

        identities = os.listdir(input_root)

        print(f"\nProcessing {split}/{subset}")

        for identity in tqdm(identities):

            src_identity_dir = os.path.join(
                input_root,
                identity
            )

            dst_identity_dir = os.path.join(
                output_root,
                identity
            )

            os.makedirs(
                dst_identity_dir,
                exist_ok=True
            )

            images = os.listdir(
                src_identity_dir
            )

            for image_name in images:

                image_path = os.path.join(
                    src_identity_dir,
                    image_name
                )

                cropped_face = crop_and_resize_face(
                    image_path
                )

                if cropped_face is None:
                    continue

                save_path = os.path.join(
                    dst_identity_dir,
                    image_name
                )

                cv2.imwrite(
                    save_path,
                    cropped_face
                )

print("Cropping completed.")

In [ ]:
#verification
for split in SPLITS:
    for subset in [
        "gallery_cropped",
        "probe_cropped"
    ]:
        total = 0
        root = os.path.join(
            ROOT,
            split,
            subset
        )
        for identity in os.listdir(root):
            total += len(
                os.listdir(
                    os.path.join(
                        root,
                        identity
                    )
                )
            )
        print(
            split,
            subset,
            total
        )